# 01 - Geração da base de leads

Base simulada para o case de priorização de leads da Captação.

- `tb_leads`: lead_id, dt_criacao, curso, modalidade, canal_origem, fl_matricula, dt_matricula
- `tb_navegacao`: session_id, lead_id, ts_evento, pagina, tempo_permanencia_seg
- `tb_abordagem_crm`: lead_id, dt_tentativa_contato, canal_contato, status_contato

Período: 180 dias até 22/11/2026. Abordagem histórica = 20% dos leads do dia por ordem de chegada.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG, SCHEMA = "cogna", "captacao"
FQ = f"{CATALOG}.{SCHEMA}"
DT_FIM = "2026-11-22"
N_DIAS = 180
N_BASE = 420_000      # antes do filtro de dia da semana / mês fica ~280k leads
CAPACIDADE = 0.20     # % dos leads do dia que o comercial consegue abordar

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
FIM_SEG = F.unix_timestamp(F.lit(DT_FIM + " 23:59:59"))

def sorteia(u, opcoes):
    # escolhe um valor a partir de um rand() usando os pesos acumulados
    acc = opcoes[0][1]
    expr = F.when(u < acc, F.lit(opcoes[0][0]))
    for v, p in opcoes[1:-1]:
        acc += p
        expr = expr.when(u < acc, F.lit(v))
    return expr.otherwise(F.lit(opcoes[-1][0]))

def de_para(col, d, padrao=0.0):
    m = F.create_map(*[F.lit(x) for kv in d.items() for x in kv])
    return F.coalesce(m[col], F.lit(padrao))

def sigmoide(x):
    return 1 / (1 + F.exp(-x))

## Leads

In [0]:
cursos = [("Administração", .16), ("Pedagogia", .13), ("Direito", .12), ("Enfermagem", .10),
          ("Psicologia", .09), ("Análise e Desenv. de Sistemas", .09), ("Ciências Contábeis", .07),
          ("Gestão de RH", .06), ("Engenharia Civil", .05), ("Educação Física", .05),
          ("Nutrição", .04), ("Marketing", .04)]
peso_curso = {"Enfermagem": .3, "Direito": .2, "Psicologia": .1, "Engenharia Civil": -.2, "Marketing": -.2}
cursos_presenciais = ["Direito", "Enfermagem", "Psicologia", "Engenharia Civil", "Educação Física", "Nutrição"]

canais = [("google_ads", .30), ("meta_ads", .25), ("organico", .20), ("afiliados", .10),
          ("indicacao", .08), ("email_mkt", .07)]
peso_canal = {"google_ads": .3, "meta_ads": -.3, "organico": .5, "indicacao": 1.2, "afiliados": -.6, "email_mkt": 0.0}
# canal vem vazio com mais frequência em orgânico/e-mail (sem utm)
perc_canal_vazio = {"organico": .60, "email_mkt": .30, "afiliados": .20, "indicacao": .10, "google_ads": .15, "meta_ads": .15}
perc_tel_invalido = {"afiliados": .25}

leads = (spark.range(N_BASE, numPartitions=32)
    .withColumn("off", F.floor(F.rand(1) * N_DIAS).cast("int"))
    .withColumn("dt", F.expr(f"date_sub(to_date('{DT_FIM}'), off)"))
    .withColumn("peso_dia", F.expr("case dayofweek(dt) when 1 then 0.45 when 7 then 0.55 when 2 then 1.0 else 0.9 end"))
    .withColumn("peso_mes", F.expr("case when month(dt) in (7, 11) then 1.0 when month(dt) in (6, 8, 10) then 0.8 else 0.7 end"))
    .filter(F.rand(2) < F.col("peso_dia") * F.col("peso_mes"))
    .withColumn("u_h", F.rand(3))
    .withColumn("hora", F.when(F.col("u_h") < .05, F.floor(F.rand(4) * 7))
                         .when(F.col("u_h") < .40, 8 + F.floor(F.rand(4) * 4))
                         .when(F.col("u_h") < .70, 13 + F.floor(F.rand(4) * 6))
                         .otherwise(19 + F.floor(F.rand(4) * 5)))
    .withColumn("ts_seg", F.unix_timestamp(F.col("dt").cast("timestamp")) + F.col("hora") * 3600 + F.floor(F.rand(5) * 3600))
    .withColumn("u_curso", F.rand(6)).withColumn("curso", sorteia(F.col("u_curso"), cursos))
    .withColumn("modalidade", F.when(F.rand(7) < F.when(F.col("curso").isin(cursos_presenciais), .40).otherwise(.85), "EAD").otherwise("Presencial"))
    .withColumn("u_canal", F.rand(8)).withColumn("canal", sorteia(F.col("u_canal"), canais))
    .withColumn("canal_origem", F.when(F.rand(9) < de_para(F.col("canal"), perc_canal_vazio), F.lit(None).cast("string")).otherwise(F.col("canal")))
    .withColumn("propensao", de_para(F.col("canal"), peso_canal) + de_para(F.col("curso"), peso_curso)
                + F.when(F.col("modalidade") == "Presencial", .2).otherwise(0.0)
                + F.when(F.col("hora") >= 19, .25).otherwise(0.0) + F.randn(10))
)
# id = L + data + sequência no dia (janela por dia, sem jogar tudo numa partição só)
leads = (leads
    .withColumn("ordem_dia", F.row_number().over(Window.partitionBy("dt").orderBy("ts_seg", "id")))
    .select(F.concat(F.lit("L"), F.date_format("dt", "yyyyMMdd"), F.lpad(F.col("ordem_dia").cast("string"), 5, "0")).alias("lead_id"),
            "dt", "ordem_dia", "ts_seg", F.timestamp_seconds("ts_seg").alias("dt_criacao"), "hora",
            "curso", "modalidade", "canal", "canal_origem", "propensao"))
leads.write.mode("overwrite").saveAsTable(f"{FQ}.stg_leads")

## Abordagem (CRM)
Por dia, os primeiros 20% que chegaram são abordados. Até 3 tentativas por lead.

In [0]:
base = (spark.table(f"{FQ}.stg_leads")
    .withColumn("leads_dia", F.count("*").over(Window.partitionBy("dt")))
    .withColumn("abordado", F.col("ordem_dia") <= F.ceil(F.col("leads_dia") * CAPACIDADE))
    .withColumn("tel_invalido", F.col("abordado") & (F.rand(11) < de_para(F.col("canal"), perc_tel_invalido, .07)))
    .withColumn("atendeu", F.col("abordado") & ~F.col("tel_invalido") & (F.rand(12) < sigmoide(-0.3 + 0.3 * F.col("propensao"))))
    .withColumn("p_mat", sigmoide(-5.0 + F.col("propensao") + F.when(F.col("atendeu"), 1.1).otherwise(0.0)))
    .withColumn("matriculou", F.rand(13) < F.col("p_mat"))
    .withColumn("t1_seg", F.col("ts_seg") + ((0.3 + F.exp(F.randn(14) * 0.8 + 0.5)) * 3600).cast("long"))
    .withColumn("intervalo_seg", ((20 + F.rand(15) * 30) * 3600).cast("long"))
    .withColumn("u_k", F.rand(16))
    .withColumn("tentativa_atend", F.when(F.col("u_k") < .6, 1).when(F.col("u_k") < .9, 2).otherwise(3))
    .withColumn("qtd_tentativas", F.when(F.col("tel_invalido"), 1).when(F.col("atendeu"), F.col("tentativa_atend")).otherwise(3))
    .withColumn("ts_atend_seg", F.col("t1_seg") + (F.col("tentativa_atend") - 1) * F.col("intervalo_seg"))
    .withColumn("u_st", F.rand(17))
    .withColumn("status_final",
        F.when(F.col("tel_invalido"), "numero_invalido")
         .when(~F.col("atendeu"), "nao_atendeu")
         .when(F.col("matriculou"), sorteia(F.col("u_st"), [("atendeu_agendou_visita", .40), ("atendeu_em_negociacao", .55), ("atendeu_sem_interesse", .05)]))
         .otherwise(sorteia(F.col("u_st"), [("atendeu_agendou_visita", .08), ("atendeu_em_negociacao", .42), ("atendeu_sem_interesse", .50)])))
    .withColumn("dias_ate_mat", F.least(F.lit(60), F.when(F.col("atendeu"), F.floor(F.exp(F.randn(18) * 0.7 + 1.4)))
                                                    .otherwise(F.floor(F.exp(F.randn(18) * 0.9 + 2.0)))))
    .withColumn("dt_mat", F.when(F.col("matriculou"),
        F.date_add(F.to_date(F.timestamp_seconds(F.when(F.col("atendeu"), F.col("ts_atend_seg")).otherwise(F.col("ts_seg")))), F.col("dias_ate_mat").cast("int"))))
    .withColumn("fl_matricula", (F.col("matriculou") & (F.col("dt_mat") <= F.lit(DT_FIM).cast("date"))).cast("int"))
    .withColumn("dt_matricula", F.when(F.col("fl_matricula") == 1, F.col("dt_mat")))
)
base.write.mode("overwrite").saveAsTable(f"{FQ}.stg_leads_crm")

In [0]:
base = spark.table(f"{FQ}.stg_leads_crm")

crm = (base.filter("abordado")
    .withColumn("n", F.explode(F.sequence(F.lit(1), F.col("qtd_tentativas"))))
    .withColumn("t_seg", F.col("t1_seg") + (F.col("n") - 1) * F.col("intervalo_seg") + F.floor(F.rand(19) * 1800))
    .filter(F.col("t_seg") <= FIM_SEG)
    .select("lead_id",
            F.timestamp_seconds("t_seg").alias("dt_tentativa_contato"),
            F.when(F.rand(20) < .6, "WhatsApp").otherwise("Telefone").alias("canal_contato"),
            F.when(F.col("n") == F.col("qtd_tentativas"), F.col("status_final")).otherwise(F.lit("nao_atendeu")).alias("status_contato")))
crm.write.mode("overwrite").saveAsTable(f"{FQ}.tb_abordagem_crm")

(base.select("lead_id", "dt_criacao", "curso", "modalidade", "canal_origem", "fl_matricula", "dt_matricula")
     .write.mode("overwrite").saveAsTable(f"{FQ}.tb_leads"))

## Navegação

In [0]:
def escolhe_pagina(u, z):
    s = sigmoide(z - 0.5)
    pesos = [("home", F.lit(.18)), ("lista_cursos", F.lit(.14)), ("pagina_curso", F.lit(.22)),
             ("blog", .04 + .12 * (1 - s)), ("ead_como_funciona", F.lit(.06)),
             ("preco_mensalidade", .08 + .10 * s), ("bolsas_descontos", .06 + .06 * s),
             ("vestibular_inscricao", .03 + .08 * s), ("contato", F.lit(.03))]
    total = sum((p for _, p in pesos[1:]), pesos[0][1])
    alvo = u * total
    acc = pesos[0][1]
    expr = F.when(alvo < acc, F.lit(pesos[0][0]))
    for v, p in pesos[1:-1]:
        acc = acc + p
        expr = expr.when(alvo < acc, F.lit(v))
    return expr.otherwise(F.lit(pesos[-1][0]))

def tempo_pagina(pagina, z, seed):
    pag_decisao = pagina.isin("preco_mensalidade", "bolsas_descontos", "vestibular_inscricao", "matricula_pagamento")
    return F.least(F.lit(1800), F.floor(F.exp(F.randn(seed) * 0.9 + 3.4 + F.when(pag_decisao, .5).otherwise(0.0) + .15 * z))).cast("int")

# sessões dos leads: 1 de cadastro + outras antes/depois
sessoes = (base
    .withColumn("qtd_sessoes", F.least(F.lit(25), 1 + F.floor(-F.log(F.rand(21)) * 2.2 * F.exp(.45 * F.col("propensao")))).cast("int"))
    .withColumn("k", F.explode(F.sequence(F.lit(1), F.col("qtd_sessoes"))))
    .withColumn("u_tipo", F.rand(22))
    .withColumn("tipo", F.when(F.col("k") == 1, "cadastro").when(F.col("u_tipo") < .4, "antes").otherwise("depois"))
    .withColumn("ini_seg", F.when(F.col("tipo") == "cadastro", F.col("ts_seg"))
                            .when(F.col("tipo") == "antes", F.col("ts_seg") - (F.rand(23) * 14 * 86400).cast("long"))
                            .otherwise(F.col("ts_seg") + (F.rand(23) * 21 * 86400).cast("long")))
    .withColumn("logado", F.when(F.col("tipo") == "depois", F.rand(24) < .9).otherwise(F.lit(False)))
    .select("lead_id", "propensao", "k", "tipo", "ini_seg", "logado"))

# acesso de quem matriculou (pagamento / área do aluno)
sessoes_mat = (base.filter("fl_matricula = 1")
    .select("lead_id", "propensao", F.lit(99).alias("k"), F.lit("matricula").alias("tipo"),
            (F.unix_timestamp(F.col("dt_matricula").cast("timestamp")) + ((10 + F.rand(25) * 10) * 3600).cast("long")).alias("ini_seg"),
            F.lit(True).alias("logado")))

eventos = (sessoes.unionByName(sessoes_mat)
    .withColumn("session_id", F.substring(F.sha2(F.concat_ws("#", "lead_id", "k", F.lit("captacao")), 256), 1, 20))
    .withColumn("qtd_ev", F.when(F.col("tipo") == "matricula", 3)
                           .otherwise(F.least(F.lit(40), 1 + F.floor(-F.log(F.rand(26)) * (3 + 1.5 * F.greatest(F.col("propensao"), F.lit(0.0))))).cast("int")
                                      + F.when(F.col("tipo") == "cadastro", 1).otherwise(0)))
    .withColumn("pos_cadastro", F.ceil(F.col("qtd_ev") * 0.6).cast("int"))
    .withColumn("i", F.explode(F.sequence(F.lit(1), F.col("qtd_ev"))))
    .withColumn("ts_ev_seg", F.when(F.col("tipo") == "cadastro", F.col("ini_seg") + (F.col("i") - F.col("pos_cadastro")) * 75)
                              .otherwise(F.col("ini_seg") + (F.col("i") - 1) * (45 + F.floor(F.rand(27) * 60))))
    .withColumn("pagina",
        F.when(F.col("tipo") == "matricula", F.element_at(F.array(F.lit("vestibular_inscricao"), F.lit("matricula_pagamento"), F.lit("area_do_aluno")), F.col("i")))
         .when((F.col("tipo") == "cadastro") & (F.col("i") == F.col("pos_cadastro")), F.lit("cadastro_lead"))
         .otherwise(escolhe_pagina(F.rand(28), F.col("propensao"))))
    .withColumn("lead_id_ev",
        F.when(F.col("tipo") == "cadastro", F.when(F.col("i") >= F.col("pos_cadastro"), F.col("lead_id")))
         .when(F.col("logado"), F.col("lead_id")))
    .filter(F.col("ts_ev_seg") <= FIM_SEG)
    .select("session_id", F.col("lead_id_ev").alias("lead_id"), F.timestamp_seconds("ts_ev_seg").alias("ts_evento"),
            "pagina", tempo_pagina(F.col("pagina"), F.col("propensao"), 29).alias("tempo_permanencia_seg")))
eventos.write.mode("overwrite").saveAsTable(f"{FQ}.tb_navegacao")

In [0]:
# visitantes que nunca viraram lead (completa ~60% de eventos sem lead_id)
qtd = spark.table(f"{FQ}.tb_navegacao").agg(F.count("*").alias("total"), F.count("lead_id").alias("ident")).first()
qtd_sessoes_anon = int(max(0, qtd.ident / 0.40 - qtd.total) / 3.6)

visitantes = (spark.range(qtd_sessoes_anon, numPartitions=64)
    .withColumn("z", F.randn(31) - 0.6)
    .withColumn("ini_seg", F.unix_timestamp(F.lit(DT_FIM).cast("timestamp")) - (F.rand(32) * N_DIAS * 86400).cast("long"))
    .withColumn("session_id", F.substring(F.sha2(F.concat_ws("#", F.lit("v"), "id", F.lit("captacao")), 256), 1, 20))
    .withColumn("qtd_ev", F.least(F.lit(40), 1 + F.floor(-F.log(F.rand(33)) * (3 + 1.5 * F.greatest(F.col("z"), F.lit(0.0))))).cast("int"))
    .withColumn("i", F.explode(F.sequence(F.lit(1), F.col("qtd_ev"))))
    .withColumn("pagina", escolhe_pagina(F.rand(34), F.col("z")))
    .select("session_id", F.lit(None).cast("string").alias("lead_id"),
            F.timestamp_seconds(F.col("ini_seg") + (F.col("i") - 1) * (45 + F.floor(F.rand(35) * 60))).alias("ts_evento"),
            "pagina", tempo_pagina(F.col("pagina"), F.col("z"), 36).alias("tempo_permanencia_seg")))
visitantes.write.mode("append").saveAsTable(f"{FQ}.tb_navegacao")

spark.sql(f"DROP TABLE IF EXISTS {FQ}.stg_leads")
spark.sql(f"DROP TABLE IF EXISTS {FQ}.stg_leads_crm")

DataFrame[]

## Conferência

In [0]:
display(spark.sql(f"""
SELECT 'tb_leads' tabela, count(*) linhas FROM {FQ}.tb_leads
UNION ALL SELECT 'tb_abordagem_crm', count(*) FROM {FQ}.tb_abordagem_crm
UNION ALL SELECT 'tb_navegacao', count(*) FROM {FQ}.tb_navegacao
"""))

tabela,linhas
tb_leads,281526
tb_abordagem_crm,123898
tb_navegacao,6466609
